In [3]:
import pandas as pd

In [5]:
df = pd.read_csv(
    '/work/ExPsyLing_E2026/LDT/CSV_LDT_priming_data.CSV',
    sep=';',
    decimal=',',
    na_values='#NULL!'
)
#Why this? Der var et problem med filen, fordi decimaler pga. den danske udgave af excel var skrevet med komma 
#og ikke punktum. Det skulle ændres. Samtidig var seperaters skrevet med ; og der hvor der ikke var noget 
#var #NULL! og det skulle den lige have at vide. 
#Basically sep=';' splits columns on semicolons unstead of normal commas. 
#decimal=',' makes pandas read the number 3,3 as 3.3. Otherwise it would be read as text. 
#na_values='#NULL!' turns it into proper missing values (NaN) without it it would be read as text. 

In [6]:
keep_default_na=False
#hvorfor? fordi et af ordene i undersøgelsen er "nan" hvilket python tolker som NaN (missing value).
#Det gør den ikke, hvis det her er tilføjet. 

In [7]:
#jeg leder efter stimuli lige nu, så jeg dropper de ikke-relevante kolonner
df = df[["prime", "target", "lexicality", "type", "rel"]]

#Stimuli bliver vist mange gange til forskellige deltagere, så jeg fjerner duplicates
df = df.drop_duplicates(subset=["prime", "target", "type", "rel"])

In [8]:
related   = df[(df["type"] == "first") & (df["rel"] == "rel")]
unrelated = df[(df["type"] == "first") & (df["rel"] == "un")]
nonword   = df[df["type"] == "nw"]

#1. df["type"] == "first" tjekker hver række og giver True, hvis kolonnen type har værdien "first", ellers False. Resultatet er en lang liste af True/False, én værdi per række.
#2. df["rel"] == "rel" gør det samme for kolonnen rel.
#3. & betyder "og". En række får kun True, hvis begge krav er opfyldt
#df[...] med den liste udenfor beholder kun de rækker, der er True. Det kaldes boolean indexing.
#Hvorfor type == "first" ? I datasættet er der også rækker med type == "other". De er ordpar, der er med i det oprindelige eksperiment, men som ikke indgår i din opgave. Når du kun vælger "first", falder de automatisk væk. Det er også grunden til, at du ikke behøver at filtrere dem fra separat.

print(len(related), len(unrelated), len(nonword))

#her tjekker jeg, om det virkede, ved at printe længden på rækkerne og se, at det er gemt rigtigt.

1661 1661 26131


In [9]:
import random

#skal hentes, så jeg kan bruge funktionen

In [10]:
n_per_condition = 10

#her viser jeg, hvor mange par jeg vil have fra hver betingelse

In [11]:
random.seed(42)

#42 er tilfældigt valgt. seed fastlægger startpunktet, så jeg får det samme hver gang jeg kører cellen.

In [12]:
def sample_pairs(cond_df, n):
    idx = list(cond_df.index)
    random.shuffle(idx)
    return cond_df.loc[idx[:n]]


#her laver jeg en funktion. cond_df er betingelsen (f.eks. "related") og n er antallet af par 
#cond_df.index er rækkernes numre (deres "etiketter") i den DataFrame. Efter filtreringen er det ikke 0, 1, 2 osv., men de oprindelige rækkenumre fra df, f.eks. 2, 5, 6, 10...
#list(...) laver rækkenumrene om til en almindelig liste, som random.shuffle() kan arbejde på.
#random.shuffle(idx) blander listen på stedet. Den returnerer ikke noget, men ændrer selve idx.
#idx[:n] tager de første n elementer fra den blandede liste. Det svarer til at trække n tilfældige rækkenumre uden tilbagelægning, så du aldrig får samme par to gange.
#cond_df.loc[...] henter de rækker, der har de rækkenumre. Det er derfor, vi bruger .loc og ikke .iloc: .loc finder rækker ud fra deres etiket.
#return sender de udvalgte rækker tilbage.

In [13]:
stimuli = pd.concat([
    sample_pairs(related, n_per_condition),
    sample_pairs(unrelated, n_per_condition),
    sample_pairs(nonword, n_per_condition),
])

#her bruges funktionen 1 gang på hver condition i n antal gange (n er her sat til 10)

In [14]:
print(len(stimuli))  
print(stimuli.groupby(["type", "rel"]).size())

#yay det virkede 

30
type   rel
first  rel    10
       un     10
nw     nw     10
dtype: int64


In [15]:
stimuli["condition"] = "nonword"

#her laver jeg en ny kolonne, condition, og giver alle i rækken værdien "nonword". Senere overskriver jeg nogle af dem.

In [16]:
stimuli.loc[stimuli["rel"] == "rel", "condition"] = "related"
stimuli.loc[stimuli["rel"] == "un", "condition"] = "unrelated"

#.loc[række-filter, kolonne] vælger de rækker, der opfylder kravet, og sætter værdien i den kolonne:
#    Rækker, hvor rel er "rel", får condition = "related"
#    Rækker, hvor rel er "un", får condition = "unrelated".
#    Nonword-rækkerne har rel == "nw", så de rammes ikke af nogen af de to linjer og beholder "nonword" fra starten.

In [17]:
print(stimuli)

             prime       target  lexicality   type  rel  condition
10566        COACH         team           1  first  rel    related
6515      MAGICIAN        magic           1  first  rel    related
12698   DISSIMILAR      similar           1  first  rel    related
8203        CLUMSY        klutz           1  first  rel    related
2710         SCREW       driver           1  first  rel    related
11639         SAVE         keep           1  first  rel    related
5995   COMPETITION        sport           1  first  rel    related
10899         ROBE         bath           1  first  rel    related
8060          FLIP         flop           1  first  rel    related
11605       STRESS      anxiety           1  first  rel    related
11945      BELIEVE         mile           1  first   un  unrelated
9276       PUBERTY       daring           1  first   un  unrelated
4768          TUBE      america           1  first   un  unrelated
4840        JEWISH       prison           1  first   un  unrel

In [18]:
stimuli[stimuli["condition"] == "related"]
#sådan ser jeg en af betingelserne 

,prime,target,lexicality,type,rel,condition
10566,COACH,team,1,first,rel,related
6515,MAGICIAN,magic,1,first,rel,related
12698,DISSIMILAR,similar,1,first,rel,related
8203,CLUMSY,klutz,1,first,rel,related
2710,SCREW,driver,1,first,rel,related
11639,SAVE,keep,1,first,rel,related
5995,COMPETITION,sport,1,first,rel,related
10899,ROBE,bath,1,first,rel,related
8060,FLIP,flop,1,first,rel,related
11605,STRESS,anxiety,1,first,rel,related


In [20]:
stimuli.to_csv("stimuli.csv", index=False)

#sådan gemmer jeg filen som CSV så jeg kan bruge den i Lab.js
#to_csv() skriver tabellen til en CSV-fil. 
#index=False sørger for, at rækkenumrene ikke bliver gemt som en ekstra kolonne, som Lab.js ellers ville vise som en navnløs kolonne.